# Setup (Colab + local compatible)

In [ ]:
# Cell 1 – Install dependencies (Colab) / imports (local)

# If you're in Colab, install packages (safe to rerun)
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q ultralytics thop pandas matplotlib scipy opencv-python

import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import cv2
import scipy.io as sio  # for .mat files from EgoHands

from ultralytics import YOLO
from thop import profile

# EXTRA para Kaggle + XML (por si las anotaciones son Pascal VOC)
import xml.etree.ElementTree as ET

try:
    import kagglehub
except ImportError:
    kagglehub = None
    print("kagglehub no está instalado. Si quieres descargar de Kaggle desde código: `pip install kagglehub`")


device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running in Colab:", IN_COLAB)
print("Using device:", device)
print("Torch version:", torch.__version__)


# Project paths and dataset config

In [ ]:
# Cell 2 – Paths and high-level configuration

# Root directory for everything
if IN_COLAB:
    PROJ_ROOT = Path("/content/hand_bbox_baseline")
else:
    PROJ_ROOT = Path.cwd() / "hand_bbox_baseline"

RAW_ROOT   = PROJ_ROOT / "data" / "raw"
WORK_ROOT  = PROJ_ROOT / "data" / "work"
YOLO_ROOT  = PROJ_ROOT / "data" / "hands_yolo"  # final YOLO dataset

RAW_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
YOLO_ROOT.mkdir(parents=True, exist_ok=True)

# EgoHands labeled data archive (pixel-level masks and frames) :contentReference[oaicite:1]{index=1}
EGO_ZIP_URL  = "https://vision.soic.indiana.edu/egohands_files/egohands_data.zip"
EGO_ZIP_PATH = RAW_ROOT / "egohands_data.zip"
EGO_EXTRACT_DIR = WORK_ROOT / "egohands_data"  # where we unzip

KAGGLE_HAND_ROOT = RAW_ROOT / "kaggle_hand_dataset"  # lo usaremos más abajo


print("Project root:", PROJ_ROOT)
print("Raw root:", RAW_ROOT)
print("Work root:", WORK_ROOT)
print("YOLO dataset root:", YOLO_ROOT)
print("Kaggle hand root (esperado):", KAGGLE_HAND_ROOT)


In [ ]:
# Cell 2b – Paths for Kaggle/Oxford Hand dataset

# Raíz donde se ha descargado el zip con kagglehub
KAGGLE_HAND_ROOT = RAW_ROOT / "kaggle_hand_dataset" / "hand_dataset"

KAGGLE_TRAIN_IMG_DIR = KAGGLE_HAND_ROOT / "training_dataset"   / "training_data"   / "images"
KAGGLE_TRAIN_ANN_DIR = KAGGLE_HAND_ROOT / "training_dataset"   / "training_data"   / "annotations"

KAGGLE_VAL_IMG_DIR   = KAGGLE_HAND_ROOT / "validation_dataset" / "validation_data" / "images"
KAGGLE_VAL_ANN_DIR   = KAGGLE_HAND_ROOT / "validation_dataset" / "validation_data" / "annotations"

KAGGLE_TEST_IMG_DIR  = KAGGLE_HAND_ROOT / "test_dataset"       / "test_data"       / "images"
KAGGLE_TEST_ANN_DIR  = KAGGLE_HAND_ROOT / "test_dataset"       / "test_data"       / "annotations"

print("KAGGLE_HAND_ROOT:", KAGGLE_HAND_ROOT)
print("Train images:", KAGGLE_TRAIN_IMG_DIR)
print("Val images:",   KAGGLE_VAL_IMG_DIR)
print("Test images:",  KAGGLE_TEST_IMG_DIR)


# Download EgoHands (only if not already)

In [ ]:
# Cell 3 – Download EgoHands labeled data (if not already downloaded)

import urllib.request

if not EGO_ZIP_PATH.exists():
    print("Downloading EgoHands labeled data...")
    urllib.request.urlretrieve(EGO_ZIP_URL, EGO_ZIP_PATH)
    print("Downloaded to:", EGO_ZIP_PATH)
else:
    print("EgoHands zip already exists:", EGO_ZIP_PATH)


In [ ]:
# Cell 3b – Descargar Hand Dataset de Kaggle (opcional)

if kagglehub is not None:
    # Esto descargará y extraerá el dataset a una carpeta cache y te devuelve la ruta
    kaggle_path = Path(kagglehub.dataset_download("armannikkhah/hand-dataset"))
    print("Kaggle dataset descargado en:", kaggle_path)

    # Si quieres tenerlo dentro de nuestro árbol de proyecto:
    if not KAGGLE_HAND_ROOT.exists():
        KAGGLE_HAND_ROOT.mkdir(parents=True, exist_ok=True)
        # Copiamos o enlazamos (aquí copio de forma simple: OJO, puede tardar un poco)
        import shutil
        shutil.copytree(kaggle_path, KAGGLE_HAND_ROOT, dirs_exist_ok=True)
        print("Copiado a:", KAGGLE_HAND_ROOT)
    else:
        print("KAGGLE_HAND_ROOT ya existe, no copiamos de nuevo:", KAGGLE_HAND_ROOT)
else:
    print("kagglehub no disponible. Descarga manualmente el zip de Kaggle y descomprímelo en:", KAGGLE_HAND_ROOT)


# Unzip EgoHands (only if not already)

In [ ]:
# Cell 4 – Unzip EgoHands labeled data (if not already extracted)

import zipfile

if not EGO_EXTRACT_DIR.exists():
    print("Extracting EgoHands zip...")
    with zipfile.ZipFile(EGO_ZIP_PATH, 'r') as zf:
        zf.extractall(EGO_EXTRACT_DIR)
    print("Extracted to:", EGO_EXTRACT_DIR)
else:
    print("EgoHands already extracted at:", EGO_EXTRACT_DIR)


# Explore Egohands structure (just to know what we have)

In [ ]:
# Cell 5 – Quick peek at EgoHands folder structure

for root, dirs, files in os.walk(EGO_EXTRACT_DIR):
    print(root)
    print("  dirs:", dirs[:5])
    print("  files:", [f for f in files if f.endswith(('.jpg', '.mat'))][:5])
    break  # just top-level


In [ ]:
# Cell 5b – Quick peek al Hand Dataset de Kaggle

if not KAGGLE_HAND_ROOT.exists():
    print("KAGGLE_HAND_ROOT no existe todavía:", KAGGLE_HAND_ROOT)
else:
    print("KAGGLE_HAND_ROOT:", KAGGLE_HAND_ROOT)
    for root, dirs, files in os.walk(KAGGLE_HAND_ROOT):
        print("root:", root)
        print("  dirs:", dirs[:10])
        print("  files:", files[:10])
        break  # just top-level



# Convert EgoHands polygons → YOLO boxes

We:

Find all folders that contain a polygons.mat.

For each folder:

  Load polygons.mat (matlab file with polygons per image).
  GitHub

  Sort the .jpg frames.

  For each image, read polygons, compute a bounding box for each hand polygon.

  Create YOLO labels (single class 0 = hand).

In [ ]:
# Cell 6 – Convert EgoHands polygons to YOLO labels (in memory)

from glob import glob

def poly_to_bbox(points):
    """
    points: array of shape (N, 2) with pixel coordinates [x, y]
    returns: (x1, y1, x2, y2) in pixel coordinates
    """
    xs = points[:, 0]
    ys = points[:, 1]
    x1, y1 = xs.min(), ys.min()
    x2, y2 = xs.max(), ys.max()
    return x1, y1, x2, y2


def bbox_to_yolo(x1, y1, x2, y2, img_w, img_h):
    """
    Convert pixel bbox to YOLO normalized cx, cy, w, h.
    """
    cx = (x1 + x2) / 2.0 / img_w
    cy = (y1 + y2) / 2.0 / img_h
    w  = (x2 - x1) / img_w
    h  = (y2 - y1) / img_h
    return cx, cy, w, h


# Gather all .mat polygons files
poly_files = sorted(glob(str(EGO_EXTRACT_DIR / "**" / "polygons.mat"), recursive=True))
print("Found polygons.mat files:", len(poly_files))

all_samples = []  # list of dicts: {image_path, boxes: [(cx,cy,w,h), ...]}

for poly_path in poly_files:
    poly_path = Path(poly_path)
    folder = poly_path.parent

    # all jpg frames in that folder
    frame_paths = sorted(glob(str(folder / "*.jpg")))
    if not frame_paths:
        continue

    # load matlab polygons
    mat = sio.loadmat(poly_path)
    # 'polygons' is a 1D cell array; polygons[0] is list over images :contentReference[oaicite:3]{index=3}
    polys = mat.get("polygons", None)
    if polys is None:
        print("WARNING: 'polygons' not found in", poly_path)
        continue

    polys = polys[0]  # shape (num_images,)

    if len(polys) != len(frame_paths):
        print(f"WARNING: polygons count {len(polys)} != frames {len(frame_paths)} in {folder}")

    # For safety, iterate over min length
    for img_idx, frame_path in enumerate(frame_paths[:len(polys)]):
        frame_path = Path(frame_path)
        img = cv2.imread(str(frame_path))
        if img is None:
            print("Could not read image:", frame_path)
            continue
        img_h, img_w = img.shape[:2]

        # polys[img_idx] is an array of length #hands, each entry Nx2 or empty
        poly_list = polys[img_idx]
        if poly_list.size == 0:
            # no hands
            continue

        boxes_yolo = []
        for p in poly_list:
            if p.size == 0:
                continue
            # p is Nx2 array
            x1, y1, x2, y2 = poly_to_bbox(p)
            cx, cy, w, h = bbox_to_yolo(x1, y1, x2, y2, img_w, img_h)

            # filter degenerate boxes
            if w <= 0 or h <= 0:
                continue
            # some sanity clamping
            cx = np.clip(cx, 0, 1)
            cy = np.clip(cy, 0, 1)
            w  = np.clip(w, 0, 1)
            h  = np.clip(h, 0, 1)

            boxes_yolo.append((cx, cy, w, h))

        if not boxes_yolo:
            continue

        all_samples.append({
            "image_path": frame_path,
            "img_w": img_w,
            "img_h": img_h,
            "boxes": boxes_yolo
        })

print("Total labeled images with at least one hand:", len(all_samples))


In [ ]:
# Cell 6b – Añadir Oxford/Kaggle Hand (subset VOCdevkit con XML Pascal VOC)

import xml.etree.ElementTree as ET

# Ruta base del VOCdevkit dentro del dataset de Kaggle
VOC_ROOT     = KAGGLE_HAND_ROOT / "evaluation_code" / "VOC2007" / "VOCdevkit" / "VOC2007"
VOC_IMG_DIR  = VOC_ROOT / "JPEGImages"
VOC_ANN_DIR  = VOC_ROOT / "Annotations"

print("VOC_ROOT:", VOC_ROOT)
print("VOC_IMG_DIR:", VOC_IMG_DIR)
print("VOC_ANN_DIR:", VOC_ANN_DIR)

def add_voc_kaggle(all_samples_list):
    """
    Lee todos los XML de Pascal VOC del Oxford Hand Dataset
    y los añade a all_samples en formato YOLO (cx,cy,w,h).
    """
    if not VOC_ANN_DIR.exists():
        print("No existe VOC_ANN_DIR, ¿está bien descomprimido el dataset?", VOC_ANN_DIR)
        return

    xml_paths = sorted(VOC_ANN_DIR.glob("*.xml"))
    print("XMLs encontrados en VOC_ANN_DIR:", len(xml_paths))

    added = 0

    for xml_path in xml_paths:
        try:
            tree = ET.parse(str(xml_path))
        except Exception as e:
            print("Error leyendo XML:", xml_path.name, "->", e)
            continue

        root = tree.getroot()

        # Nombre de la imagen (viene en el XML)
        filename_el = root.find("filename")
        if filename_el is None:
            continue
        filename = filename_el.text.strip()

        img_path = VOC_IMG_DIR / filename
        if not img_path.exists():
            # Algunas anotaciones pueden apuntar a imágenes que no están, por si acaso
            # traducimos por si falta el sufijo "_hand" o similar, pero de momento skip:
            # print("Imagen no encontrada para", xml_path.name, "->", img_path)
            continue

        # Tamaño de la imagen (lo usa Pascal VOC)
        size_el = root.find("size")
        if size_el is None:
            # Si no hay size en el XML, medimos directamente la imagen
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            img_h, img_w = img.shape[:2]
        else:
            img_w = int(size_el.find("width").text)
            img_h = int(size_el.find("height").text)

        boxes_yolo = []

        # Cada <object> suele ser una mano
        for obj in root.findall("object"):
            name_el = obj.find("name")
            if name_el is None:
                continue
            cls_name = name_el.text.strip().lower()
            if cls_name != "hand":
                # Por si hubiese otras clases, nos quedamos solo con "hand"
                continue

            bnd = obj.find("bndbox")
            if bnd is None:
                continue

            try:
                # Pascal VOC es 1-based (xmin=1 significa el primer píxel),
                # le restamos 1 para pasarlo a 0-based antes de normalizar.
                x1 = float(bnd.find("xmin").text) - 1.0
                y1 = float(bnd.find("ymin").text) - 1.0
                x2 = float(bnd.find("xmax").text) - 1.0
                y2 = float(bnd.find("ymax").text) - 1.0
            except Exception:
                continue

            # Clamp seguro (por si alguna coord se sale un poco)
            x1 = np.clip(x1, 0, img_w - 1)
            y1 = np.clip(y1, 0, img_h - 1)
            x2 = np.clip(x2, 0, img_w - 1)
            y2 = np.clip(y2, 0, img_h - 1)

            # Convertir a formato YOLO normalizado
            cx, cy, w, h = bbox_to_yolo(x1, y1, x2, y2, img_w, img_h)
            if w <= 0 or h <= 0:
                continue

            boxes_yolo.append((cx, cy, w, h))

        if not boxes_yolo:
            continue

        all_samples_list.append({
            "image_path": img_path,
            "img_w": img_w,
            "img_h": img_h,
            "boxes": boxes_yolo
        })
        added += 1

    print(f"Imágenes añadidas desde VOC Kaggle: {added}")


print(f"Muestras antes de VOChand (solo EgoHands): {len(all_samples)}")
add_voc_kaggle(all_samples)
print(f"Total labeled images (EgoHands + VOC Kaggle): {len(all_samples)}")


# Train/val/test split and write YOLO structure

In [ ]:
# Cell 7 – Split into train/val/test and write YOLO dataset (EgoHands + Kaggle)

from shutil import copy2
import shutil
import random
import yaml

random.seed(42)

# all_samples ya tiene EgoHands + VOC Kaggle
n_total = len(all_samples)
random.shuffle(all_samples)

n_train = int(0.8 * n_total)
n_val   = int(0.1 * n_total)
n_test  = n_total - n_train - n_val

train_samples = all_samples[:n_train]
val_samples   = all_samples[n_train:n_train+n_val]
test_samples  = all_samples[n_train+n_val:]

print(f"TOTAL samples: {n_total}")
print(f"Train: {len(train_samples)}, Val: {len(val_samples)}, Test: {len(test_samples)}")

# 1) Borramos YOLO_ROOT por completo
if YOLO_ROOT.exists():
    print("Borrando YOLO_ROOT anterior:", YOLO_ROOT)
    shutil.rmtree(YOLO_ROOT)

# 2) Creamos carpetas desde cero
for split in ["train", "val", "test"]:
    (YOLO_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)


def write_split(samples, split):
    img_out_dir = YOLO_ROOT / "images" / split
    lbl_out_dir = YOLO_ROOT / "labels" / split

    for idx, s in enumerate(samples):
        src_img_path = Path(s["image_path"])
        # Nombre único por split + índice
        new_stem = f"{split}_{idx:06d}"
        dst_img_path = img_out_dir / (new_stem + src_img_path.suffix.lower())
        copy2(src_img_path, dst_img_path)

        dst_lbl_path = lbl_out_dir / (new_stem + ".txt")
        with open(dst_lbl_path, "w") as f:
            for (cx, cy, w, h) in s["boxes"]:
                f.write(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")


write_split(train_samples, "train")
write_split(val_samples,   "val")
write_split(test_samples,  "test")

print("YOLO dataset written to:", YOLO_ROOT)

# 3) Crear/reescribir data.yaml
data_yaml = {
    "path": str(YOLO_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "hand"},
}

DATA_YAML_PATH = YOLO_ROOT / "data.yaml"
with open(DATA_YAML_PATH, "w") as f:
    yaml.dump(data_yaml, f)

print("Created data.yaml at:", DATA_YAML_PATH)


In [ ]:
# Debug: comprobar realmente lo que hay escrito en disco

from pathlib import Path

print("Comprobando contenido real de YOLO_ROOT:", YOLO_ROOT)

for split in ["train", "val", "test"]:
    img_dir = YOLO_ROOT / "images" / split
    lbl_dir = YOLO_ROOT / "labels" / split

    num_imgs = (
        len(list(img_dir.glob("*.jpg")))
        + len(list(img_dir.glob("*.png")))
        + len(list(img_dir.glob("*.jpeg")))
    )
    num_lbls = len(list(lbl_dir.glob("*.txt")))

    print(f"{split}: imgs={num_imgs}, labels={num_lbls}")


In [ ]:
# Cell 7b – Create data.yaml for YOLO

data_yaml = {
    "path": str(YOLO_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "hand"},
}

DATA_YAML_PATH = YOLO_ROOT / "data.yaml"
with open(DATA_YAML_PATH, "w") as f:
    yaml.dump(data_yaml, f)

print("Created data.yaml at:", DATA_YAML_PATH)


In [ ]:
# Cell 7c – Visual sanity check: show some train/test images with GT boxes

import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def load_yolo_boxes(label_path, img_w, img_h):
    """
    Lee un .txt en formato YOLO (cls cx cy w h) y devuelve
    una lista de cajas en pixeles: (x1, y1, x2, y2).
    """
    boxes = []
    if not label_path.exists():
        return boxes

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, cx, cy, w, h = parts
            cx = float(cx); cy = float(cy)
            w  = float(w);  h  = float(h)

            x1 = int((cx - w / 2.0) * img_w)
            y1 = int((cy - h / 2.0) * img_h)
            x2 = int((cx + w / 2.0) * img_w)
            y2 = int((cy + h / 2.0) * img_h)

            # clamp por seguridad
            x1 = max(0, min(img_w - 1, x1))
            y1 = max(0, min(img_h - 1, y1))
            x2 = max(0, min(img_w - 1, x2))
            y2 = max(0, min(img_h - 1, y2))

            boxes.append((x1, y1, x2, y2))
    return boxes


def visualize_split(split="train", n=3):
    """
    Muestra n imágenes aleatorias de 'train' o 'test'
    con sus cajas de ground-truth dibujadas.
    """
    img_dir = YOLO_ROOT / "images" / split
    lbl_dir = YOLO_ROOT / "labels" / split

    # EgoHands son .jpg, pero por si acaso miramos varios formatos
    img_paths = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpeg"))
    if not img_paths:
        print(f"No se encontraron imágenes en {img_dir}")
        return

    n = min(n, len(img_paths))
    sample_paths = random.sample(img_paths, n)

    for img_path in sample_paths:
        img = cv2.imread(str(img_path))
        if img is None:
            print("No se pudo leer la imagen:", img_path)
            continue

        img_h, img_w = img.shape[:2]

        label_path = lbl_dir / (img_path.stem + ".txt")
        boxes = load_yolo_boxes(label_path, img_w, img_h)

        # Dibujar cajas
        img_draw = img.copy()
        for (x1, y1, x2, y2) in boxes:
            cv2.rectangle(img_draw, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img_draw, "hand", (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)

        img_rgb = cv2.cvtColor(img_draw, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(6, 6))
        plt.imshow(img_rgb)
        plt.title(f"{split.upper()} - {img_path.name} (#{len(boxes)} cajas)")
        plt.axis("off")
        plt.show()

        print(f"{split} image: {img_path.name}, num_boxes = {len(boxes)}, label file = {label_path}")


# Mostrar 3 de train y 3 de test
visualize_split("train", n=3)
visualize_split("test",  n=3)


# Common training configuration + experiment list

In [ ]:
# Cell 8 – Experiments configuration (YOLOv8n with different LRs)

DATA_YAML = str(DATA_YAML_PATH)

# Where YOLO training runs will be saved
PROJECT_ROOT = PROJ_ROOT / "runs_yolo"
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# Common hyperparameters
IMG_SIZE   = 416
BATCH_SIZE = 16  # adjust for your GPU
EPOCHS     = 100

BASE_LR = 1e-3
LOW_LR  = 1e-4  # teacher's "very low" LR

EXPERIMENTS = [
    {
        "name": "yolov8n_mix_base_lr",
        "model_path": "yolov8n.pt",  # pretrained weights
        "lr0": BASE_LR,
    },
    {
        "name": "yolov8n_mix_low_lr",
        "model_path": "yolov8n.pt",
        "lr0": LOW_LR,
    },
]

print("Dataset yaml:", DATA_YAML)
print("Experiments:", [e["name"] for e in EXPERIMENTS])


# Train YOLOv8n for each experiment

In [ ]:
# Cell 9 – Training helper and running experiments

def train_experiment(exp_cfg):
    """
    Train a YOLOv8 model for a given experiment configuration.
    """
    name = exp_cfg["name"]
    model_path = exp_cfg["model_path"]
    lr0 = exp_cfg["lr0"]

    print(f"\n=== Training experiment: {name} ===")
    print(f"Model: {model_path} | lr0={lr0} | epochs={EPOCHS} | batch={BATCH_SIZE} | imgsz={IMG_SIZE}")

    model = YOLO(model_path)

    results = model.train(
        data=DATA_YAML,
        imgsz=IMG_SIZE,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        lr0=lr0,
        optimizer="AdamW",
        momentum=0.9,
        patience=25,  # stop if no val improvement for 25 epochs
        project=str(PROJECT_ROOT),
        name=name,
        exist_ok=True,
        verbose=True
    )

    run_dir = Path(results.save_dir)
    print("Run directory:", run_dir)
    return model, run_dir


trained_models = {}
run_dirs = {}

for exp in EXPERIMENTS:
    model, run_dir = train_experiment(exp)
    trained_models[exp["name"]] = model
    run_dirs[exp["name"]] = run_dir


# Utilities: box conversion + IoU + greedy matching

In [ ]:
# Cell 10 – Box utilities: convert YOLO normalized and IoU

def yolo_xywhn_to_xyxyn(cx, cy, w, h):
    x1 = cx - w / 2.0
    y1 = cy - h / 2.0
    x2 = cx + w / 2.0
    y2 = cy + h / 2.0
    return np.array([x1, y1, x2, y2], dtype=np.float32)


def box_iou_xyxy(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h

    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])
    union = area_a + area_b - inter
    if union <= 0.0:
        return 0.0
    return inter / union


def greedy_match_boxes(pred_boxes, gt_boxes, iou_min=0.1):
    """
    Greedy match PM ↔ GT based on IoU.
    """
    matches = []
    if len(pred_boxes) == 0 or len(gt_boxes) == 0:
        return matches

    pred_used = set()
    gt_used = set()

    iou_matrix = np.zeros((len(pred_boxes), len(gt_boxes)), dtype=np.float32)
    for i, pb in enumerate(pred_boxes):
        for j, gb in enumerate(gt_boxes):
            iou_matrix[i, j] = box_iou_xyxy(pb, gb)

    while True:
        i, j = np.unravel_index(np.argmax(iou_matrix), iou_matrix.shape)
        best_iou = iou_matrix[i, j]
        if best_iou < iou_min:
            break
        if i in pred_used or j in gt_used:
            iou_matrix[i, j] = -1.0
            continue

        matches.append((i, j, best_iou))
        pred_used.add(i)
        gt_used.add(j)
        iou_matrix[i, :] = -1.0
        iou_matrix[:, j] = -1.0

    return matches


# Load GT boxes from YOLO val labels

In [ ]:
# Cell 11 – Load GT boxes (normalized xyxy) for a given image

VAL_IMAGES_DIR = YOLO_ROOT / "images" / "val"
VAL_LABELS_DIR = YOLO_ROOT / "labels" / "val"

def load_gt_boxes_for_image(image_path, labels_dir=VAL_LABELS_DIR):
    image_path = Path(image_path)
    label_path = Path(labels_dir) / (image_path.stem + ".txt")
    if not label_path.exists():
        return []

    gt_boxes = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, cx, cy, w, h = parts
            cx, cy, w, h = map(float, (cx, cy, w, h))
            box_xyxy = yolo_xywhn_to_xyxyn(cx, cy, w, h)
            gt_boxes.append(box_xyxy)
    return gt_boxes


# Extra metrics: IoU, MAE, MSE, R²

In [ ]:
# Cell 12 – Evaluate extra metrics on validation set

def evaluate_extra_metrics(model, conf_threshold=0.25, max_images=None):
    all_gt_coords = []
    all_pred_coords = []
    all_ious = []

    results_gen = model.predict(
        source=str(VAL_IMAGES_DIR),
        imgsz=IMG_SIZE,
        conf=conf_threshold,
        iou=0.5,
        stream=True,
        verbose=False
    )

    for idx, r in enumerate(results_gen):
        if max_images is not None and idx >= max_images:
            break

        img_path = r.path
        if r.boxes is None or r.boxes.xyxy.shape[0] == 0:
            continue

        pred_xyxy_px = r.boxes.xyxy.cpu().numpy()
        h, w = r.orig_shape

        pred_boxes_norm = []
        for bx in pred_xyxy_px:
            x1, y1, x2, y2 = bx
            pred_boxes_norm.append(
                np.array([x1 / w, y1 / h, x2 / w, y2 / h], dtype=np.float32)
            )

        gt_boxes_norm = load_gt_boxes_for_image(img_path)

        matches = greedy_match_boxes(pred_boxes_norm, gt_boxes_norm, iou_min=0.1)

        for (pi, gi, iou) in matches:
            pb = pred_boxes_norm[pi]
            gb = gt_boxes_norm[gi]
            all_ious.append(iou)
            all_pred_coords.append(pb)
            all_gt_coords.append(gb)

    if len(all_gt_coords) == 0:
        print("No matches found; check thresholds/paths.")
        return None

    all_gt = np.stack(all_gt_coords, axis=0)
    all_pred = np.stack(all_pred_coords, axis=0)
    all_ious = np.array(all_ious)

    mae = np.mean(np.abs(all_pred - all_gt))
    mse = np.mean((all_pred - all_gt)**2)

    y_true = all_gt.reshape(-1)
    y_pred = all_pred.reshape(-1)
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")

    metrics = {
        "mean_iou": float(np.mean(all_ious)),
        "mae": float(mae),
        "mse": float(mse),
        "r2": float(r2),
        "num_matches": len(all_gt_coords),
    }
    return metrics


extra_metrics = {}
for exp in EXPERIMENTS:
    name = exp["name"]
    model = trained_models[name]
    print(f"\n=== Extra metrics for {name} ===")
    m = evaluate_extra_metrics(model, conf_threshold=0.25, max_images=None)
    print(m)
    extra_metrics[name] = m


# Training curves (loss & mAP)

In [ ]:
# Cell 13 – Plot training curves from results.csv

%matplotlib inline

def plot_training_curves(run_dir, title=None):
    csv_path = Path(run_dir) / "results.csv"
    if not csv_path.exists():
        print("No results.csv at", csv_path)
        return

    df = pd.read_csv(csv_path)
    epochs = df["epoch"]

    plt.figure(figsize=(10, 6))

    if "train/box_loss" in df.columns:
        plt.plot(epochs, df["train/box_loss"], label="train box loss")
    if "val/box_loss" in df.columns:
        plt.plot(epochs, df["val/box_loss"], label="val box loss")
    if "metrics/mAP50(B)" in df.columns:
        plt.plot(epochs, df["metrics/mAP50(B)"], label="mAP50(B)")
    if "metrics/mAP50-95(B)" in df.columns:
        plt.plot(epochs, df["metrics/mAP50-95(B)"], label="mAP50-95(B)")

    plt.xlabel("Epoch")
    plt.legend()
    plt.grid(True)
    plt.title(title if title else str(run_dir))
    plt.tight_layout()
    plt.show()


for exp in EXPERIMENTS:
    name = exp["name"]
    run_dir = run_dirs[name]
    print(f"\n=== Curves for {name} ===")
    plot_training_curves(run_dir, title=name)


# Params, FLOPs, latency CPU/GPU

In [ ]:
# Cell 14 – Params, FLOPs (MACs), latency

def count_params(model):
    # model: objeto YOLO de Ultralytics
    total = sum(p.numel() for p in model.model.parameters())
    trainable = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
    return total, trainable



def compute_flops(model, img_size=IMG_SIZE):
    dummy = torch.randn(1, 3, img_size, img_size).to(device)
    model.model.to(device)
    model.model.eval()
    flops, params = profile(model.model, inputs=(dummy,), verbose=False)
    return flops, params


def benchmark_latency(model, img_size=IMG_SIZE, device_str="cuda", n_warmup=10, n_iters=50):
    dev = torch.device(device_str)
    model.model.to(dev)
    model.model.eval()

    dummy = torch.randn(1, 3, img_size, img_size, device=dev)

    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model.model(dummy)
        if device_str == "cuda":
            torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(n_iters):
            t0 = time.perf_counter()
            _ = model.model(dummy)
            if device_str == "cuda":
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            times.append(t1 - t0)

    return np.mean(times) * 1000.0  # ms


model_stats = {}

for exp in EXPERIMENTS:
    name = exp["name"]
    model = trained_models[name]
    print(f"\n=== Model stats for {name} ===")


    total_params, trainable_params = count_params(model)
    params_m = total_params / 1e6

    print(f"Params: {params_m:.3f} M ({total_params:,} total)")
    print(f"Trainable params (if any): {trainable_params:,}")

    try:
        flops, params_thop = compute_flops(model, IMG_SIZE)
        flops_g = flops / 1e9
        print(f"Approx MACs: {flops_g:.3f} G")
    except Exception as e:
        print("FLOPs estimation failed:", e)
        flops_g = None

    gpu_ms = None
    if torch.cuda.is_available():
        gpu_ms = benchmark_latency(model, IMG_SIZE, device_str="cuda")
        print(f"GPU latency: {gpu_ms:.2f} ms / img")

    cpu_ms = benchmark_latency(model, IMG_SIZE, device_str="cpu")
    print(f"CPU latency: {cpu_ms:.2f} ms / img")

    model_stats[name] = {
        "params_m": params_m,
        "flops_gmacs": flops_g,
        "gpu_ms": gpu_ms,
        "cpu_ms": cpu_ms,
    }


# Final comparison table (for TFG)

In [ ]:
# Cell 15 – Build and export comparison table

rows = []

for exp in EXPERIMENTS:
    name = exp["name"]
    stats = model_stats.get(name, {})
    extra = extra_metrics.get(name, {})

    run_dir = run_dirs[name]
    csv_path = Path(run_dir) / "results.csv"
    map50 = None
    map5095 = None
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        if "metrics/mAP50(B)" in df.columns:
            map50 = df["metrics/mAP50(B)"].max()
        if "metrics/mAP50-95(B)" in df.columns:
            map5095 = df["metrics/mAP50-95(B)"].max()

    row = {
        "experiment": name,
        "params_M": stats.get("params_m"),
        "FLOPs_GMACs": stats.get("flops_gmacs"),
        "GPU_ms_per_img": stats.get("gpu_ms"),
        "CPU_ms_per_img": stats.get("cpu_ms"),
        "mAP50": map50,
        "mAP50_95": map5095,
        "mean_IoU": extra.get("mean_iou"),
        "MAE_coords": extra.get("mae"),
        "MSE_coords": extra.get("mse"),
        "R2_coords": extra.get("r2"),
        "num_matches": extra.get("num_matches"),
    }
    rows.append(row)

df_summary = pd.DataFrame(rows)
df_summary


In [ ]:
# Still Cell 15 – Save to CSV
summary_path = PROJ_ROOT / "hands_model_comparison.csv"
df_summary.to_csv(summary_path, index=False)
print("Saved comparison table to:", summary_path)


# Inferencia con el mejor modelo + visualización

In [ ]:
# Cell 16 – Inferencia con el mejor modelo y visualización de bounding boxes

# 1. Especificar manualmente la ruta al modelo
# ---------------------------------------------
# CAMBIA ESTA RUTA para apuntar a tu archivo best.pt
best_ckpt_path = Path("hand_bbox_baseline/runs_yolo/yolov8n_mix_base_lr/weights/best.pt")

# Alternativamente, si quieres usar uno de los modelos entrenados en este notebook:
# best_ckpt_path = run_dirs["yolov8n_base_lr"] / "weights" / "best.pt"
# best_ckpt_path = run_dirs["yolov8n_low_lr"] / "weights" / "best.pt"

print(f"Usando checkpoint: {best_ckpt_path}")

# 2. Verificar que el archivo existe
if not best_ckpt_path.exists():
    print(f"ERROR: No se encuentra el archivo {best_ckpt_path}")
    print("\nModelos entrenados disponibles en este notebook:")
    for name, rd in run_dirs.items():
        ckpt = Path(rd) / "weights" / "best.pt"
        print(f"  - {name}: {ckpt} (existe: {ckpt.exists()})")
else:
    # 3. Cargar el modelo YOLO desde ese checkpoint
    best_model = YOLO(str(best_ckpt_path))

    # 4. Elegir una imagen para probar
    img_path = Path("fototest1.jpeg")
    print(f"Imagen seleccionada: {img_path}")

    # 5. Hacer la inferencia con el modelo
    results = best_model(str(img_path))

    # 6. Visualizar la imagen con las cajas predichas
    res = results[0]
    im_bgr = res.plot()
    im_rgb = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(6, 6))
    plt.imshow(im_rgb)
    plt.title(f"Predicción - {best_ckpt_path.stem}")
    plt.axis("off")
    plt.show()

    # 7. (Opcional) Imprimir info numérica de las predicciones
    print("\nPredicciones (bbox normalizadas xywh, conf, cls):")
    for box in res.boxes:
        xywhn = box.xywhn.cpu().numpy().ravel()
        conf = float(box.conf.cpu().numpy().ravel()[0])
        cls  = int(box.cls.cpu().numpy().ravel()[0])
        print(f"  cls={cls}, conf={conf:.3f}, xywhn={xywhn}")
